In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2


from src.data import load_fd001, add_rul, drop_cols

In [19]:
import numpy as np
import pandas as pd
train, test, rul = load_fd001()
train = add_rul(train)
print(train.shape)
print(train.groupby("unit")["RUL"].min().unique())
print(train["RUL"].max())

(20631, 27)
[0]
125


GroupKFold:

In [3]:
from sklearn.model_selection import GroupKFold
features = ["s2", "s3", "s4", "s8", "s11", "s13", "s15", "s17", "s7", "s12", "s20", "s21", "s9", "s14"]
X = train[features]
y = train["RUL"]
groups = train["unit"]
gfk = GroupKFold(n_splits = 10)

In [4]:
splits = list(gfk.split(X,y, groups = groups))
print(len(splits))
print(splits[0])

10
(array([    0,     1,     2, ..., 20628, 20629, 20630], shape=(18559,)), array([ 4565,  4566,  4567, ..., 19008, 19009, 19010], shape=(2072,)))


In [5]:
for fold, (train_idx, val_idx) in enumerate(splits):
    train_units = set(groups.iloc[train_idx])
    val_units = set(groups.iloc[val_idx])
    overlap = train_units & val_units
    print(f"Fold {fold}: {len(val_units)} Triebwerke, {len(val_idx)} Zeilen, Überschneidung {len(overlap)}")

Fold 0: 10 Triebwerke, 2072 Zeilen, Überschneidung 0
Fold 1: 10 Triebwerke, 2057 Zeilen, Überschneidung 0
Fold 2: 10 Triebwerke, 2069 Zeilen, Überschneidung 0
Fold 3: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 4: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 5: 10 Triebwerke, 2055 Zeilen, Überschneidung 0
Fold 6: 10 Triebwerke, 2048 Zeilen, Überschneidung 0
Fold 7: 10 Triebwerke, 2056 Zeilen, Überschneidung 0
Fold 8: 10 Triebwerke, 2071 Zeilen, Überschneidung 0
Fold 9: 10 Triebwerke, 2071 Zeilen, Überschneidung 0


Formel für NASA-Score: 
$$s = \begin{cases} e^{-d/13} - 1 & \text{wenn } d < 0 \quad \text{(zu früh)} \\ e^{d/10} - 1 & \text{wenn } d \geq 0 \quad \text{(zu spät)} \end{cases}$$
mit d = Vorhersage - wahre RUL

In [6]:
print(f"10 Zyklen zu früh: {np.exp(10/13)-1:.2f} 30 Zyklen zu früh: {np.exp(30/13)-1:.2f}")
print(f"10 Zyklen zu spät: {np.exp(10/10)-1:.2f} 30 Zyklen zu spät: {np.exp(30/10)-1:.2f}")

10 Zyklen zu früh: 1.16 30 Zyklen zu früh: 9.05
10 Zyklen zu spät: 1.72 30 Zyklen zu spät: 19.09


In [11]:
from src.metrics import nasa_score

nasa_score([50],[40])

np.float64(1.1581055339484458)

In [12]:
nasa_score([50, 50], [40, 60])

np.float64(2.876387362407491)

In [13]:
nasa_score([50,50], [40,60], "mean")

np.float64(1.4381936812037455)

In [15]:
nasa_score([50,50],[50,50])

np.float64(0.0)

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

ergebnisse = []

for fold, (train_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LinearRegression()
    model.fit(X_tr,y_tr)
    pred_tr = model.predict(X_tr)
    pred_val = model.predict(X_val)

    ergebnisse.append({

        "fold": fold,
        "rmse_train": root_mean_squared_error(y_tr, pred_tr),
        "rmse_val": root_mean_squared_error(y_val, pred_val),
        "nasa_val": nasa_score(y_tr, pred_tr, "mean"),
    })

cv = pd.DataFrame(ergebnisse)
cv.mean(), cv.std()
cv.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val
mean,21.485017,21.560367,9.318112
std,0.149752,1.333467,0.284705


In [23]:
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error

ergebnisse = []

for fold, (train_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBRegressor()
    model.fit(X_tr,y_tr)
    pred_tr = model.predict(X_tr)
    pred_val = model.predict(X_val)

    ergebnisse.append({

        "fold": fold,
        "rmse_train": root_mean_squared_error(y_tr, pred_tr),
        "rmse_val": root_mean_squared_error(y_val, pred_val),
        "nasa_val": nasa_score(y_tr, pred_tr, "mean"),
    })

cv = pd.DataFrame(ergebnisse)
cv.mean(), cv.std()
cv.drop(columns="fold").agg(["mean", "std"])

,rmse_train,rmse_val,nasa_val
mean,11.950677,19.225841,2.960536
std,0.188642,2.442047,0.124311
